In [1]:
import pandas as pd
import os

In [2]:
os.getcwd()

'/data/clusterfs/mld/users/moukan/automatic_annotation_gaze'

In [3]:
data = pd.read_csv("/data/workspaces/mld/workspaces/mld-kanakanti-shared/analysis/automatic_annotation_gaze/datasets/MyFix1/submission/data/gaze_positions.csv")

In [4]:
data

,gaze_timestamp,world_index,confidence,norm_pos_x,norm_pos_y,base_data,gaze_point_3d_x,gaze_point_3d_y,gaze_point_3d_z,eye_center0_3d_x,...,eye_center0_3d_z,gaze_normal0_x,gaze_normal0_y,gaze_normal0_z,eye_center1_3d_x,eye_center1_3d_y,eye_center1_3d_z,gaze_normal1_x,gaze_normal1_y,gaze_normal1_z
0,0.351773,0,1.0,0.560235,0.191360,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.355780,0,1.0,0.563847,0.194312,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.359758,0,1.0,0.564937,0.198029,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.363767,1,1.0,0.565262,0.197967,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.371736,1,1.0,0.567744,0.198756,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353643,1774.779741,53243,1.0,0.648147,0.318140,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
353644,1774.787784,53243,1.0,0.648508,0.317863,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
353645,1774.791731,53243,1.0,0.648624,0.317376,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
353646,1774.795767,53244,1.0,0.649013,0.316186,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
converted_data = data[['norm_pos_x', 'norm_pos_y', 'confidence']]

In [6]:
converted_data = converted_data.rename(columns={"norm_pos_x": "x", "norm_pos_y": "y"})

In [7]:
converted_data

,x,y,confidence
0,0.560235,0.191360,1.0
1,0.563847,0.194312,1.0
2,0.564937,0.198029,1.0
3,0.565262,0.197967,1.0
4,0.567744,0.198756,1.0
...,...,...,...
353643,0.648147,0.318140,1.0
353644,0.648508,0.317863,1.0
353645,0.648624,0.317376,1.0
353646,0.649013,0.316186,1.0


In [8]:
def _detect_coordinate_type(df):
    """
    Detect coordinate system type
    """
    # Sample for efficiency
    sample_size = min(1000, len(df))
    sample_df = df.sample(n=sample_size) if len(df) > sample_size else df
    
    x_min, x_max = sample_df['x'].min(), sample_df['x'].max()
    y_min, y_max = sample_df['y'].min(), sample_df['y'].max()
    
    # Check for [-1,1] range (centered)
    if x_min >= -1.1 and x_max <= 1.1 and y_min >= -1.1 and y_max <= 1.1:
        if x_min < -0.1 or y_min < -0.1:  # Has negative values
            return 'centered'
    
    # Check for [0,1] range (normalized)
    if x_max <= 1.1 and y_max <= 1.1 and x_min >= -0.1 and y_min >= -0.1:
        return 'norm_tl'  # Assume top-left origin by default
    
    # Otherwise assume pixel coordinates
    return 'pixel'

def _convert_to_pixels(df, coord_system, frame_width, frame_height):
    """
    Convert various coordinate systems to pixel coordinates (top-left origin)
    """
    df = df.copy()
    
    if coord_system == 'norm_tl':
        # [0,1] range, top-left origin
        df['x'] = df['x'] * frame_width
        df['y'] = df['y'] * frame_height
        
    elif coord_system == 'norm_bl':
        # [0,1] range, bottom-left origin
        df['x'] = df['x'] * frame_width
        df['y'] = (1 - df['y']) * frame_height
        
    elif coord_system == 'centered':
        # [-1,1] range, center origin - assume top-left output
        df['x'] = ((df['x'] + 1) / 2) * frame_width
        df['y'] = ((df['y'] + 1) / 2) * frame_height
    
    print(f"Converted {coord_system} coordinates to pixels ({frame_width}x{frame_height})")
    return df

In [9]:
_detect_coordinate_type(converted_data)

'norm_tl'

In [10]:
pixel_df = _convert_to_pixels(converted_data, 'norm_tl', 1088, 1080)

Converted norm_tl coordinates to pixels (1088x1080)


In [11]:
pixel_df

,x,y,confidence
0,609.535339,206.668884,1.0
1,613.465393,209.857178,1.0
2,614.651306,213.871521,1.0
3,615.004517,213.804199,1.0
4,617.705078,214.655945,1.0
...,...,...,...
353643,705.183472,343.591003,1.0
353644,705.576843,343.291992,1.0
353645,705.702576,342.765869,1.0
353646,706.126038,341.481262,1.0


In [12]:
pixel_df.to_csv('/data/workspaces/mld/workspaces/mld-kanakanti-shared/analysis/automatic_annotation_gaze/datasets/MyFix1/test/worldTrim.csv')

In [13]:
labels = pd.read_csv("/data/workspaces/mld/workspaces/mld-kanakanti-shared/analysis/automatic_annotation_gaze/datasets/MyFix1/submission/evaluation/labeled_data.csv")

In [14]:
import os
from PIL import Image
import numpy as np
import torch
from sam2.build_sam import build_sam2, build_sam2_video_predictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

In [ ]:
# Setup
video_path = "notebooks/videos/bedroom"
sam2_checkpoint = "checkpoints/sam2_hiera_tiny.pt"
model_cfg = "sam2_hiera_t.yaml"
device = "cuda"

In [ ]:




# Auto-masking of first frame (from automatic mask generation notebook)
sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)
first_frame_path = os.path.join(video_path, os.listdir(video_path)[0])
first_frame = Image.open(first_frame_path)
first_frame = np.array(first_frame.convert("RGB"))
mask_generator = SAM2AutomaticMaskGenerator(sam2)
auto_masks = mask_generator.generate(first_frame)
print("Number of auto-masks:", len(auto_masks))

# Add every 'auto-mask' as it's own prompt for video tracking
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
inference_state = predictor.init_state(video_path=video_path)
dtype = next(predictor.parameters()).dtype
lowres_side_length = predictor.image_size // 4
for mask_idx, mask_result in enumerate(auto_masks):

    # Get mask into form expected by the model
    mask_tensor = torch.tensor(mask_result["segmentation"], dtype=dtype, device=device)
    lowres_mask = torch.nn.functional.interpolate(
        mask_tensor.unsqueeze(0).unsqueeze(0),
        size=(lowres_side_length, lowres_side_length),
        mode="bilinear",
        align_corners=False,
    ).squeeze()

    # Add each mask as it's own 'object' to segment
    _, out_obj_ids, out_mask_logits = predictor.add_new_mask(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=mask_idx,
        mask=lowres_mask,
    )

# Do video segmentation (same as video segmentation notebook)
video_segments = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy() for i, out_obj_id in enumerate(out_obj_ids)
    }

In [2]:
import pandas as pd
import numpy as np

def convert_txt_to_csv(input_file, output_file):
    """
    Convert TXT data to CSV with columns: frame, x, y, label
    Groups by frame and averages x,y coordinates, takes most common label
    
    Args:
        input_file (str): Path to input TXT file
        output_file (str): Path to output CSV file
    """
    
    # Read the data from TXT file
    df = pd.read_csv(input_file, sep='\t')
    
    # Create dataframe with required columns
    data = pd.DataFrame({
        'frame': df['Frame'],
        'x': pd.to_numeric(df['X'], errors='coerce'),
        'y': pd.to_numeric(df['Y'], errors='coerce'),
        'label': df['AOI']
    })
    
    # Remove rows where frame is NaN
    data = data.dropna(subset=['frame'])
    
    # Group by frame and aggregate
    csv_data = data.groupby('frame').agg({
        'x': 'mean',          # Average x coordinates
        'y': 'mean',          # Average y coordinates  
        'label': lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]  # Most common label
    }).reset_index()
    
    # Round coordinates to reasonable precision
    csv_data['x'] = csv_data['x'].round(6)
    csv_data['y'] = csv_data['y'].round(6)
    
    # Save to CSV
    csv_data.to_csv(output_file, index=False)
    
    print(f"Conversion completed! CSV saved as: {output_file}")
    print(f"Total frames: {len(csv_data)}")
    print(f"Columns: {list(csv_data.columns)}")
    
    # Display first few rows
    print("\nFirst 10 rows of converted data:")
    print(csv_data.head(10))
    
    return csv_data

def convert_from_string(data_string):
    """
    Convert TXT data from string format to CSV
    Groups by frame and averages x,y coordinates, takes most common label
    
    Args:
        data_string (str): Raw text data as string
    
    Returns:
        pandas.DataFrame: Converted data
    """
    
    # Split into lines and process
    lines = data_string.strip().split('\n')
    
    # Extract header
    header = lines[0].split('\t')
    
    # Process data rows
    data_rows = []
    for line in lines[1:]:
        row = line.split('\t')
        data_rows.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=header)
    
    # Create dataframe with required columns
    data = pd.DataFrame({
        'frame': df['Frame'],
        'x': pd.to_numeric(df['X'], errors='coerce'),
        'y': pd.to_numeric(df['Y'], errors='coerce'),
        'label': df['AOI']
    })
    
    # Remove rows where frame is NaN
    data = data.dropna(subset=['frame'])
    
    # Group by frame and aggregate
    csv_data = data.groupby('frame').agg({
        'x': 'mean',          # Average x coordinates
        'y': 'mean',          # Average y coordinates  
        'label': lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]  # Most common label
    }).reset_index()
    
    # Round coordinates to reasonable precision
    csv_data['x'] = csv_data['x'].round(6)
    csv_data['y'] = csv_data['y'].round(6)
    
    return csv_data

if __name__ == "__main__":
    # Method 1: Convert from file
    convert_txt_to_csv('/data/workspaces/mld/workspaces/mld-kanakanti-shared/analysis/automatic_annotation_gaze/datasets/AutomaticAOIconstruction/AutomaticAOIconstruction/output/testvideo_allDataAOI.txt', 
    '/data/workspaces/mld/workspaces/mld-kanakanti-shared/analysis/automatic_annotation_gaze/datasets/AutomaticAOIconstruction/AutomaticAOIconstruction/output/testvideo_allDataAOI.csv')

Conversion completed! CSV saved as: /data/workspaces/mld/workspaces/mld-kanakanti-shared/analysis/automatic_annotation_gaze/datasets/AutomaticAOIconstruction/AutomaticAOIconstruction/output/testvideo_allDataAOI.csv
Total frames: 1640
Columns: ['frame', 'x', 'y', 'label']

First 10 rows of converted data:
   frame         x           y     label
0    1.0  543.6500  305.713333  RightEye
1    2.0  540.0300  309.222000  RightEye
2    3.0  541.2525  312.497500  RightEye
3    4.0  546.3425  318.150000  RightEye
4    5.0  545.7800  320.987500  RightEye
5    6.0  546.1225  313.790000  RightEye
6    7.0  546.4940  302.568000  RightEye
7    8.0  547.5875  303.232500  RightEye
8    9.0  544.4875  316.872500  RightEye
9   10.0  587.7400  366.192500  RightEye


In [1]:
import os
os.getcwd()

'/data/clusterfs/mld/users/moukan/automatic_annotation_gaze/core'